# QT / corrected QT study

Questions: does the QT interval (and heart-rate corrected QTc) **correlate with age and with the median-beat groups**, and **could it help the grouping**?

**Measurement** (`qt_interval.py`, on the **500 Hz** records for 2 ms precision; median heartbeat from −300 to +700 ms so a long QT is not cut off)
- QRS onset: where the 12-lead QRS magnitude rises above 15% of its peak
- T end: tangent method (steepest slope after the T peak, extended to the PR baseline), median over the leads with a clear T wave (I, II, V2–V6)
- **Watch QT**: the same measurement on lead I only, to see whether a watch could provide it
- QTc by four formulas: Bazett QT/√RR, Fridericia QT/∛RR, Framingham QT + 154(1 − RR), Hodges QT + 1.75(HR − 60)

**Validation**: PTB-XL's long-QT label (`LNGQT`, 117 ECGs); QTc should not depend on heart rate; women should have a longer QTc than men.

**Excluded**: pacemaker, atrial fibrillation / flutter (QT is unreliable with irregular RR), ECGs without a clear T wave, QT outside 200–700 ms.

**Run `model/ml.ipynb` first**: this notebook loads the trained watch model (`model/watch_model.joblib`: lead-I groups,
risk-model settings) and the 12-lead clusters (`model/median_beat_clusters.csv`) from it, and recomputes the rest.

In [ ]:
# setup: libraries, the trained watch model, chart style
import ast
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import wfdb
from scipy.stats import kruskal, spearmanr
from scipy.stats import t as t_dist
from sklearn.cluster import KMeans
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

sys.path.insert(0, os.path.join("..", "model"))
from watch_pipeline import SUPERCLASSES, WatchModel, rhythm_features, watch_beat, watch_rr
from qt_interval import measure_qt

# PTB-XL folder in the project root (git-ignored; a symlink to the dataset works)
ecg_db_path = os.path.join("..", "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3")
wm = WatchModel.load(os.path.join("..", "model", "watch_model.joblib"))
clusters = pd.read_csv(os.path.join("..", "model", "median_beat_clusters.csv"), index_col=0)  # 12-lead clusters

superclasses = SUPERCLASSES
WATCH_LEAD = 0
TRAIN_FOLDS, TEST_FOLDS = list(range(1, 9)), [9, 10]
AGE_BANDS = wm.b["step2"]["age_bands"]
band_names = [f"{lo}-{hi - 1}" if hi < 120 else f"{lo}+" for lo, hi in zip(AGE_BANDS[:-1], AGE_BANDS[1:])]

INK, INK_2, MUTED, GRID_COLOR, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#fcfcfb"
CLUSTER_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # categorical slots 1-4
SC_COLORS = dict(zip(superclasses, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]))  # slots 1-5
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "axes.edgecolor": "#c3c2b7",
    "axes.labelcolor": INK_2, "xtick.color": MUTED, "ytick.color": MUTED, "text.color": INK,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "axes.grid.axis": "y",
    "grid.color": GRID_COLOR, "grid.linewidth": 0.6, "font.size": 9, "axes.titlesize": 10, "axes.axisbelow": True,
})

In [ ]:
# patient metadata (same preparation as model/ml.ipynb)
Y = pd.read_csv(os.path.join(ecg_db_path, "ptbxl_database.csv"), index_col="ecg_id")
Y.scp_codes = Y.scp_codes.apply(ast.literal_eval)
Y.recording_date = pd.to_datetime(Y.recording_date, format="%Y-%m-%d %H:%M:%S")
Y.loc[Y.age == 300, "age"] = np.nan  # ages over 89 are stored as 300

agg_df = pd.read_csv(os.path.join(ecg_db_path, "scp_statements.csv"), index_col=0)
agg_df = agg_df[agg_df.diagnostic == 1]
Y["diagnostic_superclass"] = Y.scp_codes.apply(
    lambda codes: sorted({agg_df.loc[c, "diagnostic_class"] for c in codes if c in agg_df.index}))
Y["visit_number"] = Y.sort_values(["patient_id", "recording_date"]).groupby("patient_id").cumcount() + 1

# every patient's first ECG + both ECGs of the patients with exactly two ECGs
ecgs_per_patient = Y.groupby("patient_id").size()
two_ecg = Y[Y.patient_id.isin(ecgs_per_patient[ecgs_per_patient == 2].index)].sort_values(["patient_id", "recording_date"])
first_ids = Y.index[Y.visit_number == 1]
study_ids = first_ids.union(two_ecg.index)
print(f"{len(first_ids)} patients, {len(study_ids)} ECGs in the study")

In [ ]:
# lead-I (watch) beats and groups from the trained model: train folds 1-8 = the model's own groups, folds 9-10 assigned
read_lead1 = lambda eid: wfdb.rdsamp(os.path.join(ecg_db_path, Y.filename_lr[eid]), channels=[WATCH_LEAD])[0][:, 0]
lead1 = {eid: read_lead1(eid) for eid in study_ids}
w_beats = {eid: watch_beat(sig, 100) for eid, sig in lead1.items()}
w_usable = pd.Series({eid: t is not None and t["usable"] for eid, t in w_beats.items()})
w_usable &= Y.loc[w_usable.index, "pacemaker"].isna() & Y.loc[w_usable.index, "electrodes_problems"].isna()

peers = wm.b["peers"]  # training patients and their groups, saved with the model
w_train_ids = [e for e in first_ids[Y.loc[first_ids, "strat_fold"].isin(TRAIN_FOLDS)] if w_usable[e]]
w_test_ids = [e for e in first_ids[Y.loc[first_ids, "strat_fold"].isin(TEST_FOLDS)] if w_usable[e]]
assert w_train_ids == list(peers.index), "lead-I processing differs from model/ml.ipynb"
w_train_groups = peers.group.values
unit_vec = {e: wm.assign_group(w_beats[e]["beat"][:, 0])[1] for e in w_usable.index[w_usable]}
w_test_groups = np.array([wm.assign_group(w_beats[e]["beat"][:, 0])[0] for e in w_test_ids])
W_train = np.stack([unit_vec[e] for e in w_train_ids])
W_K = len(wm.b["step1"]["profile"])
has = lambda ids, sc: Y.loc[ids, "diagnostic_superclass"].apply(lambda x: sc in x)

# a person's two recordings (test folds, both usable) for the stability check
test_pairs = pd.DataFrame({"ecg_1": two_ecg.groupby("patient_id").head(1).index.values,
                           "ecg_2": two_ecg.groupby("patient_id").tail(1).index.values}).assign(pair="same patient")
test_pairs = test_pairs[Y.loc[test_pairs.ecg_1, "strat_fold"].isin(TEST_FOLDS).values
                        & test_pairs.ecg_1.map(w_usable).values & test_pairs.ecg_2.map(w_usable).values]
print(f"lead-I groups: train {len(w_train_ids)}, test {len(w_test_ids)} patients; {len(test_pairs)} test patients with two recordings")

In [ ]:
# step-3 risk-model features and settings (for the "does QT help the risk model" check)
s3 = wm.b["step3"]
feature_cols = s3["feature_cols"]
best = pd.DataFrame(s3["params"]).T.loc[superclasses]
PARAM_NAMES = list(best.columns)


def user_from_Y(eid):
    p_ = Y.loc[eid]
    return {"age": p_.age, "sex": "female" if p_.sex == 1 else "male",
            "height_cm": p_.height if p_.height == p_.height else None,
            "weight_kg": p_.weight if p_.weight == p_.weight else None, "diagnosed_label": None}


def hgb(**params):
    params = {k: int(v) if k in ("max_leaf_nodes", "min_samples_leaf") else float(v) for k, v in params.items()}
    return HistGradientBoostingClassifier(max_iter=1000, early_stopping=True, validation_fraction=0.15,
                                          n_iter_no_change=20, random_state=0, **params)


risk_ids = [e for e in w_train_ids + w_test_ids if len(Y.loc[e, "diagnostic_superclass"]) > 0]
F_all = pd.DataFrame.from_dict({e: wm.feature_row(w_beats[e], rhythm_features(watch_rr(lead1[e], 100)), user_from_Y(e))
                                for e in risk_ids}, orient="index")
L_all = pd.DataFrame({sc: Y.loc[risk_ids, "diagnostic_superclass"].apply(lambda x: sc in x) for sc in superclasses}).astype(int)
folds = Y.loc[risk_ids, "strat_fold"].values
is_tr, is_val = np.isin(folds, TRAIN_FOLDS), folds == 9
print(f"risk-model features: train {is_tr.sum()}, validation (fold 9) {is_val.sum()}")

In [ ]:
# measure QT on every patient's first ECG + both ECGs of the 2-ECG patients (12-lead and lead I = watch)
from scipy.stats import kruskal, spearmanr
from scipy.stats import t as t_dist
from qt_interval import measure_qt

QT_FS = 500
qt_rows = {}
for eid in study_ids:
    sig = wfdb.rdsamp(os.path.join(ecg_db_path, Y.filename_hr[eid]))[0]
    m12 = measure_qt(sig, QT_FS)
    mw = measure_qt(sig[:, WATCH_LEAD], QT_FS, t_leads=(0,))
    row = {} if m12 is None else dict(m12)
    row |= {} if mw is None else {f"watch_{k}": v for k, v in mw.items()}
    qt_rows[eid] = row
qt_df = pd.DataFrame.from_dict(qt_rows, orient="index")

codes_all = Y.loc[qt_df.index, "scp_codes"]
qt_df["long_qt_label"] = codes_all.apply(lambda c: "LNGQT" in c)
qt_df["excluded"] = (codes_all.apply(lambda c: bool({"AFIB", "AFLT", "PACE"} & set(c)))
                     | Y.loc[qt_df.index, "pacemaker"].notna())
QTC_FORMULAS = ["qtc_bazett", "qtc_fridericia", "qtc_framingham", "qtc_hodges"]
for prefix in ["", "watch_"]:
    bad = qt_df["excluded"] | ~qt_df[f"{prefix}qt_ms"].between(200, 700)
    qt_df.loc[bad, [f"{prefix}qt_ms"] + [f"{prefix}{f}" for f in QTC_FORMULAS]] = np.nan
qt_df = qt_df.join(Y[["age", "sex", "diagnostic_superclass", "strat_fold"]])

qt_first = qt_df.loc[qt_df.index.intersection(first_ids)]
print(f"{len(qt_first)} first ECGs: excluded (pacemaker / AF) {qt_first.excluded.sum()}, "
      f"QT measured {qt_first.qt_ms.notna().sum()} (12-lead), {qt_first.watch_qt_ms.notna().sum()} (lead I)")
qt_first[["qt_ms", "heart_rate"] + QTC_FORMULAS + ["watch_qt_ms"]].describe(percentiles=[0.05, 0.5, 0.95]).round(1).T

In [ ]:
# validate: which correction removes the heart-rate dependence, does QTc find long QT, sex difference, watch agreement
ref = qt_first[~qt_first.long_qt_label]
val_rows = {}
for f in ["qt_ms"] + QTC_FORMULAS:
    ok = qt_first[f].notna()
    val_rows[f] = {
        "Spearman rho vs RR (should be ~0)": spearmanr(ref[f], ref.rr_s, nan_policy="omit")[0],
        "long-QT AUC": roc_auc_score(qt_first.long_qt_label[ok], qt_first[f][ok]),
        "median, men (ms)": ref.loc[ref.sex == 0, f].median(),
        "median, women (ms)": ref.loc[ref.sex == 1, f].median(),
    }
qt_validation = pd.DataFrame(val_rows).T
qt_validation["women - men (ms)"] = qt_validation["median, women (ms)"] - qt_validation["median, men (ms)"]
QTC = qt_validation.loc[QTC_FORMULAS, "Spearman rho vs RR (should be ~0)"].abs().idxmin()  # best rate correction
WATCH_QTC = f"watch_{QTC}"

both = qt_first[[QTC, WATCH_QTC]].dropna()
diff = both[WATCH_QTC] - both[QTC]
print(f"using {QTC} | watch (lead I) vs 12-lead {QTC}: n={len(both)}, r={np.corrcoef(both[QTC], both[WATCH_QTC])[0, 1]:.3f}, "
      f"bias {diff.mean():+.1f} ms, median |difference| {diff.abs().median():.1f} ms, within 20 ms: {(diff.abs() <= 20).mean():.0%}")
qt_validation.round(3)

In [ ]:
# QTc vs age: overall, by sex, and in normal ECGs only (healthy ageing vs disease)
def ols(y, X_):
    """Linear regression with 95% CIs and p-values (no statsmodels needed)."""
    Xm = np.column_stack([np.ones(len(X_)), X_.values.astype(float)])
    beta, *_ = np.linalg.lstsq(Xm, y, rcond=None)
    dof = len(y) - Xm.shape[1]
    resid = y - Xm @ beta
    se = np.sqrt(np.diag(resid @ resid / dof * np.linalg.inv(Xm.T @ Xm)))
    return pd.DataFrame({"coef": beta, "95% CI low": beta - 1.96 * se, "95% CI high": beta + 1.96 * se,
                         "p": 2 * t_dist.sf(np.abs(beta / se), dof)}, index=["intercept"] + list(X_.columns))


qa = qt_first.dropna(subset=[QTC, "age"])
is_norm = qa.diagnostic_superclass.apply(lambda x: x == ["NORM"])
age_rows = {}
for name, d in [("all", qa), ("men", qa[qa.sex == 0]), ("women", qa[qa.sex == 1]), ("normal ECGs only", qa[is_norm]),
                ("abnormal ECGs", qa[~is_norm])]:
    rho, p_ = spearmanr(d.age, d[QTC])
    slope = np.polyfit(d.age, d[QTC], 1)[0] * 10
    age_rows[name] = {"n": len(d), "Spearman rho": rho, "p": p_, "ms per decade": slope}
qt_age = pd.DataFrame(age_rows).T
qt_by_age = qa.groupby([pd.cut(qa.age, AGE_BANDS, right=False), qa.sex.map({0: "men", 1: "women"})], observed=False)[QTC] \
    .median().unstack()
display(qt_age.round(4), qt_by_age.round(1))

In [ ]:
# QTc by group: lead-I (watch) groups, 12-lead clusters, diagnostic superclass; and adjusted for age + sex
def epsilon_sq(h, n):
    return h / (n - 1)  # Kruskal-Wallis effect size: share of rank variance explained by the grouping


lead1_group = pd.Series(np.r_[w_train_groups, w_test_groups], index=list(w_train_ids) + list(w_test_ids), name="lead-I group")
qg = qt_first.join(lead1_group).join(clusters["cluster"].rename("12-lead cluster"))
group_rows = []
for col in ["lead-I group", "12-lead cluster"]:
    d = qg.dropna(subset=[QTC, col])
    samples = [g[QTC].values for _, g in d.groupby(col)]
    h, p_ = kruskal(*samples)
    group_rows.append({"grouping": col, "n": len(d), "Kruskal-Wallis H": h, "p": p_, "epsilon^2": epsilon_sq(h, len(d)),
                       **{f"median {col.split()[0]} {int(k)}": v for k, v in d.groupby(col)[QTC].median().items()}})
qt_groups = pd.DataFrame(group_rows).set_index("grouping")

qt_by_class = pd.DataFrame({sc: qg.loc[qg.diagnostic_superclass.apply(lambda x: sc in x), QTC].describe()[["count", "50%"]]
                            for sc in superclasses}).T.rename(columns={"50%": f"median {QTC}"})

# does the lead-I group still matter after age + sex? (group 1 = reference)
d = qg.dropna(subset=[QTC, "age", "lead-I group"])
X_adj = pd.concat([d[["age"]], d.sex.rename("female"),
                   pd.get_dummies(d["lead-I group"].astype(int), prefix="group", drop_first=True).astype(float)], axis=1)
qt_adjusted = ols(d[QTC].values, X_adj)
display(qt_groups.round(4), qt_by_class.round(1), qt_adjusted.round(4))

In [ ]:
# could QTc help grouping? cluster lead-I beats + weight * z(QTc), compare with beats only (weight 0)
# measured on the test folds: how informative the groups are about diagnosis, and how stable a person's two recordings are
own_test = test_pairs[test_pairs.pair == "same patient"]


def grouping_metrics(weight, qtc_col):
    mu, sd = qt_df.loc[w_train_ids, qtc_col].mean(), qt_df.loc[w_train_ids, qtc_col].std()
    z = lambda ids: ((qt_df[qtc_col].reindex(ids) - mu) / sd).fillna(0).values[:, None]  # missing QTc -> average
    km = KMeans(W_K, n_init=10, random_state=0).fit(np.hstack([W_train, weight * z(w_train_ids)]))
    lab = lambda ids: km.predict(np.hstack([np.stack([unit_vec[e] for e in ids]), weight * z(ids)]))
    test_lab = lab(w_test_ids)
    aucs = {}
    for sc in superclasses:  # score each test patient by their group's training prevalence of the class
        prev = pd.Series(has(w_train_ids, sc).values).groupby(km.labels_).mean()
        aucs[sc] = roc_auc_score(has(w_test_ids, sc).values, prev.reindex(test_lab).values)
    qtc_test = qt_df[QTC].reindex(w_test_ids).values
    ok = ~np.isnan(qtc_test)
    h = kruskal(*[qtc_test[ok & (test_lab == g)] for g in np.unique(test_lab)])[0]
    return {"weight": weight, "QTc used": qtc_col, "diagnosis macro AUC": np.mean(list(aucs.values())),
            **{f"{sc} AUC": v for sc, v in aucs.items()},
            "same group, own 2 recordings %": np.mean(lab(own_test.ecg_1.values) == lab(own_test.ecg_2.values)) * 100,
            "QTc epsilon^2 across groups": epsilon_sq(h, ok.sum())}


QT_WEIGHTS = [0, 0.1, 0.2, 0.3, 0.5, 1.0]
qt_grouping = pd.DataFrame([grouping_metrics(w, col) for col in [WATCH_QTC, QTC] for w in QT_WEIGHTS])
qt_grouping.round(3)

In [ ]:
# could QT help the step-3 risk model? same tuned settings; train folds 1-8, compare on validation fold 9 (fold 10 untouched)
QT_FEATS = {"+ watch QT (lead I)": ["watch_qt_ms", WATCH_QTC], "+ 12-lead QT": ["qt_ms", QTC]}
F_qt = F_all.join(qt_df[sum(QT_FEATS.values(), [])])
risk_qt_rows = {}
for name, cols in {"step-3 features": feature_cols, **{k: feature_cols + v for k, v in QT_FEATS.items()}}.items():
    risk_qt_rows[name] = {}
    for sc in superclasses:
        m = hgb(**{k: best.loc[sc, k] for k in PARAM_NAMES}).fit(F_qt.loc[is_tr, cols], L_all.loc[is_tr, sc])
        risk_qt_rows[name][sc] = roc_auc_score(L_all.loc[is_val, sc], m.predict_proba(F_qt.loc[is_val, cols])[:, 1])
risk_qt = pd.DataFrame(risk_qt_rows)
risk_qt.loc["macro"] = risk_qt.mean()
risk_qt.round(4)

In [ ]:
# plots: validation, QTc vs age, QTc by group, and whether QTc helps grouping
fig, axes = plt.subplots(2, 3, figsize=(17, 9.5), constrained_layout=True)

ax = axes[0, 0]  # QT vs RR, with the chosen correction
d = qt_first.dropna(subset=["qt_ms"])
ax.hexbin(d.rr_s, d.qt_ms, gridsize=45, cmap="Blues", mincnt=1, linewidths=0)
rr_line = np.linspace(0.45, 1.5, 50)
for level in [380, 420, 460]:
    ax.plot(rr_line, level * np.cbrt(rr_line) if QTC == "qtc_fridericia" else level * np.sqrt(rr_line), color=INK_2, linewidth=1)
    ax.text(1.5, level * (np.cbrt(1.5) if QTC == "qtc_fridericia" else np.sqrt(1.5)), f" {QTC.split('_')[1]} {level}", color=INK_2, fontsize=8, va="center")
ax.set_xlabel("RR interval (s)"); ax.set_ylabel("QT (ms)")
ax.set_xlim(0.4, 1.75)
ax.set_title("QT vs RR (12-lead); lines = equal QTc", loc="left")

ax = axes[0, 1]  # correction quality
v = qt_validation.loc[QTC_FORMULAS]
x = np.arange(len(v))
ax.bar(x - 0.2, v["Spearman rho vs RR (should be ~0)"].abs(), width=0.38, color="#86b6ef", label="|rho| with RR (lower = better)")
ax.bar(x + 0.2, v["long-QT AUC"], width=0.38, color="#2a78d6", label="long-QT label AUC")
for xi, (a, b) in enumerate(zip(v["Spearman rho vs RR (should be ~0)"].abs(), v["long-QT AUC"])):
    ax.text(xi - 0.2, a + 0.01, f"{a:.2f}", ha="center", fontsize=8, color=INK_2)
    ax.text(xi + 0.2, b + 0.01, f"{b:.2f}", ha="center", fontsize=8, color=INK_2)
ax.set_xticks(x, [f.split("_")[1] for f in QTC_FORMULAS])
ax.set_ylim(0, 1.2)
ax.set_title("QT correction formulas", loc="left")
ax.legend(frameon=False, loc="upper left", ncols=2, fontsize=8)

ax = axes[0, 2]  # QTc vs age by sex
for sex_name, color in [("men", "#2a78d6"), ("women", "#eb6834")]:
    ax.plot(band_names, qt_by_age[sex_name].values, color=color, linewidth=2, marker="o", markersize=8, label=sex_name)
ax.set_xlabel("age band"); ax.set_ylabel(f"median {QTC} (ms)")
ax.set_title(f"QTc rises with age (rho = {qt_age.loc['all', 'Spearman rho']:.2f})", loc="left")
ax.legend(frameon=False)

for ax, col, title in [(axes[1, 0], "lead-I group", "QTc by lead-I (watch) group"),
                       (axes[1, 1], "diagnostic_superclass", "QTc by diagnostic superclass")]:
    if col == "lead-I group":
        cats = list(range(1, W_K + 1))
        data = [qg.loc[qg[col] == g, QTC].dropna() for g in cats]
        labels, colors = [f"group {g}" for g in cats], CLUSTER_COLORS
    else:
        cats = superclasses
        data = [qg.loc[qg.diagnostic_superclass.apply(lambda x: sc in x), QTC].dropna() for sc in cats]
        labels, colors = cats, [SC_COLORS[sc] for sc in cats]
    data = [s.clip(*np.percentile(np.concatenate(data), [1, 99])) for s in data]
    parts = ax.violinplot(data, showmedians=True, showextrema=False)
    for body, color in zip(parts["bodies"], colors):
        body.set_facecolor(color); body.set_edgecolor(color); body.set_alpha(0.5)
    parts["cmedians"].set_color(INK)
    ax.set_xticks(range(1, len(cats) + 1), labels)
    ax.set_ylabel(f"{QTC} (ms)")
    ax.set_title(title, loc="left")

ax = axes[1, 2]  # does QTc help grouping?
for col, color, name in [(QTC, "#2a78d6", "12-lead QTc"), (WATCH_QTC, "#eb6834", "watch (lead I) QTc")]:
    d = qt_grouping[qt_grouping["QTc used"] == col]
    ax.plot(d.weight, d["diagnosis macro AUC"], color=color, linewidth=2, marker="o", markersize=8, label=f"{name}: diagnosis AUC")
    ax.plot(d.weight, d["same group, own 2 recordings %"] / 100, color=color, linewidth=2, marker="o", markersize=8,
            linestyle="--", label=f"{name}: same-group share")
ax.set_xlabel("QTc weight in clustering (0 = beat shape only)")
ax.set_title("Does adding QTc help the grouping? (test)", loc="left")
ax.legend(frameon=False, fontsize=8, loc="lower left")
ax.grid(axis="x", visible=True)
fig.suptitle("QT / corrected QT study", x=0.01, ha="left", fontsize=12)
fig.savefig("qt_study.png", dpi=120)
plt.show()

#### QT / corrected QT study: results

**Measurement** (first ECG per patient): 1,417 excluded (pacemaker / AF / flutter); QT measured on 17,212 12-lead and 16,333 lead-I ECGs. Median QT 355 ms, QTc (Fridericia) 373 ms.

**Validation**

| | Spearman ρ with RR (want ~0) | Long-QT label AUC | Women − men |
|---|---|---|---|
| QT (uncorrected) | 0.64 | 0.78 | +2 ms |
| Bazett | −0.36 (over-corrects) | 0.89 | +11 ms |
| **Fridericia** | **0.02** | 0.88 | +8 ms |
| Framingham | −0.08 | 0.88 | +8 ms |
| Hodges | −0.05 | 0.88 | +8 ms |

- **Fridericia** removes the heart-rate dependence best and is used below. All corrections find PTB-XL's long-QT label well (AUC 0.88), and women have a longer QTc than men, as expected physiologically.
- The tangent method gives QTc about 30 ms below typical clinical values (median 373 vs ~400–410 ms). This is a known systematic offset of the method; comparisons between ages and groups are not affected, but the values should not be read against clinical cut-offs (e.g. 450 / 470 ms).
- **Watch (lead I) QTc**: median difference from 12-lead 10 ms, 72% within 20 ms, but r = 0.45 because of outliers (flat T waves in lead I). Usable as a trend, not yet as a clinical measurement.

**QTc vs age**: rises **~4 ms per decade** (ρ = 0.20, p < 0.001); 5.1 ms/decade in men, 3.3 in women. It also rises in normal ECGs only (2.6 ms/decade), so part of it is healthy ageing, not just disease. Women are longer at every age, by 6.5–14 ms (largest under 30, narrowing with age).

**QTc vs group**

| Grouping | Median QTcF per group | Effect size ε² |
|---|---|---|
| Lead-I (watch) groups 1 → 4 | 360 · 375 · 379 · 385 ms | **0.13** |
| 12-lead clusters 1 → 4 | 367 · 376 · 377 · 378 ms | 0.03 |
| Superclass | NORM 368 · MI 380 · STTC 385 · CD 379 · HYP 379 ms | |

- QTc rises from group 1 to 4, and much more clearly across the lead-I groups than the 12-lead clusters.
- **After adjusting for age and sex**, groups 2 / 3 / 4 still have a **+18 / +21 / +27 ms** longer QTc than group 1 (all p < 0.001); age adds 2.2 ms per decade and female sex 6.5 ms.

**Does QTc help the grouping? No.**
- Clustering lead-I beats + QTc: at small weights (0.1) the groups barely change (diagnosis macro AUC 0.708 → 0.710, same-group share 74% → 74–75%); at larger weights the groups split on QTc instead and become **less** informative about diagnosis (0.60–0.66) and less stable (55–64%).
- Adding QT / QTc to the step-3 risk model (validation fold 9): macro AUC 0.845 → 0.843 (watch QT) / 0.843 (12-lead QT): no gain.
- Why: the groups and the risk model already use the whole median beat, which contains the T-wave timing; QTc is one number summarising information they already have (that is also why QTc differs so clearly between the lead-I groups).

**Where QTc is useful instead**: as an interpretable, reportable measure (QTc per recording and its trend over time), and as a long-QT flag (AUC 0.88 on 12-lead) - for a watch it would first need validating on real watch recordings.

**Output**: `qt_study.png`; measurement code in `qt_interval.py`